# SurfaceGuard AI — Google Colab GPU Acceleration & Multi-Category Benchmarking

**Deep Learning-Based Industrial Defect Detection & Anomaly Localization**

This notebook orchestrates GPU-accelerated training, feature-bank generation, and benchmarking across the MVTec AD dataset using the modular `surfaceguard` library.

### Workflow Steps:
1. Check GPU environment and CUDA capabilities
2. (Optional) Mount Google Drive for persistent artifact & dataset storage
3. Clone/install `surfaceguard` package dependencies
4. Validate dataset directory hierarchy
5. Run data-loading and model smoke tests
6. Build feature memory banks (PatchCore / ResNet-18/50) or train Conv-Autoencoders
7. Execute multi-category evaluation
8. Export serialized memory banks, thresholds, and benchmark reports directly to Google Drive

In [ ]:
# 1. GPU Environment & System Check
import os
import sys
import torch

print("=== COMPUTE ENVIRONMENT ===")
print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Active GPU: {gpu_name} ({gpu_mem:.2f} GB VRAM)")
else:
    print("Warning: CUDA GPU not detected. Running on CPU (slower feature extraction).")

In [ ]:
# 2. Mount Google Drive for Persistent Storage
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_MOUNTED = True
    print("Google Drive successfully mounted.")
except Exception as e:
    DRIVE_MOUNTED = False
    print(f"Not running in Colab or Drive mount skipped: {e}")

In [ ]:
# 3. Project Configuration & Python Path Setup
# Set this to your cloned repository location or Google Drive folder
PROJECT_ROOT = os.path.abspath(".")
if DRIVE_MOUNTED and os.path.exists("/content/drive/MyDrive/SurfaceGuard-AI"):
    PROJECT_ROOT = "/content/drive/MyDrive/SurfaceGuard-AI"

print(f"Project Root: {PROJECT_ROOT}")
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

# Ensure dependencies are installed
!pip install -q pyyaml pydantic scikit-learn opencv-python pillow

In [ ]:
# 4. Dataset Configuration & Validation
from surfaceguard.config import ALL_CATEGORIES, resolve_path
from surfaceguard.data.validation import validate_mvtec_dataset

# Configure dataset directory path
DATASET_DIR = os.path.join(PROJECT_ROOT, "data/mvtec_ad")
if DRIVE_MOUNTED and os.path.exists("/content/drive/MyDrive/mvtec_ad"):
    DATASET_DIR = "/content/drive/MyDrive/mvtec_ad"

print(f"Validating dataset at: {DATASET_DIR}")
summary = validate_mvtec_dataset(DATASET_DIR)
print(f"Available Categories ({len(summary['available_categories'])}/15):", summary["available_categories"])
if summary["missing_categories"]:
    print(f"Missing Categories:", summary["missing_categories"])

In [ ]:
# 5. Smoke Test: Single Batch Forward Pass
from torch.utils.data import DataLoader
from surfaceguard.data.dataset import MVTecDataset
from surfaceguard.models.anomaly_detector import PatchCoreDetector

test_cat = summary["available_categories"][0] if summary["available_categories"] else "bottle"
print(f"Running smoke test on category '{test_cat}'...")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
detector = PatchCoreDetector(category=test_cat, backbone_name="resnet18", device=device, sampling_ratio=0.05)

print("Feature extractor initialized successfully on", device)

In [ ]:
# 6. Execution: Build Feature Banks & Calibrate Thresholds
# Resumes automatically: skips categories where feature bank already exists!
from surfaceguard.utils.artifacts import ArtifactManager

artifacts_mgr = ArtifactManager(os.path.join(PROJECT_ROOT, "artifacts"))
target_categories = summary["available_categories"]

print(f"Processing {len(target_categories)} categories...")

for cat in target_categories:
    bank_path = artifacts_mgr.get_feature_bank_path(cat)
    if bank_path.exists():
        print(f"[SKIP] Memory bank already exists for '{cat}' at {bank_path}")
        continue
    
    print(f"[BUILD] Building memory bank for '{cat}'...")
    train_ds = MVTecDataset(category=cat, split="train", data_dir=DATASET_DIR)
    val_ds = MVTecDataset(category=cat, split="val", data_dir=DATASET_DIR)
    
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=False)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False) if len(val_ds) > 0 else None
    
    cat_detector = PatchCoreDetector(category=cat, backbone_name="resnet18", device=device, sampling_ratio=0.10)
    bank = cat_detector.fit(train_loader)
    cat_detector.save_memory_bank(bank_path)
    
    # Threshold calibration on normal val samples
    val_scores = []
    if val_loader:
        for batch in val_loader:
            for img in batch["image"]:
                score, _ = cat_detector.predict_sample(img)
                val_scores.append(score)
    thresh = cat_detector.calibrate_threshold(val_scores, target_fpr=0.05)
    artifacts_mgr.save_threshold(cat, "patchcore", thresh)
    print(f"[DONE] Bank saved: {bank.shape} | Threshold: {thresh:.4f}")

In [ ]:
# 7. Benchmark & Evaluation
from surfaceguard.evaluation.evaluator import Evaluator
from surfaceguard.evaluation.aggregation import aggregate_benchmark_results, generate_macro_summary

results = []
for cat in target_categories:
    bank_path = artifacts_mgr.get_feature_bank_path(cat)
    if not bank_path.exists():
        continue
    
    print(f"Evaluating '{cat}'...")
    det = PatchCoreDetector(category=cat, device=device)
    det.load_memory_bank(bank_path)
    thresh = artifacts_mgr.load_threshold(cat, "patchcore")
    if thresh:
        det.calibrated_threshold = thresh
        
    test_ds = MVTecDataset(category=cat, split="test", data_dir=DATASET_DIR)
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)
    
    evaluator = Evaluator(det)
    res = evaluator.evaluate(test_loader)
    results.append(res)

df = aggregate_benchmark_results(results)
print(df)
csv_out = os.path.join(PROJECT_ROOT, "results/reports/benchmark_summary.csv")
os.makedirs(os.path.dirname(csv_out), exist_ok=True)
df.to_csv(csv_out, index=False)
print(f"Saved results to {csv_out}")

In [ ]:
# 8. Export Artifacts to Google Drive
if DRIVE_MOUNTED:
    export_dest = "/content/drive/MyDrive/SurfaceGuard_Artifacts"
    os.makedirs(export_dest, exist_ok=True)
    !cp -r "{PROJECT_ROOT}/artifacts" "{export_dest}/"
    !cp -r "{PROJECT_ROOT}/results" "{export_dest}/"
    print(f"All model checkpoints, banks, and results backed up to: {export_dest}")